In [ ]:
# ============================================================
# CELL 1 - Imports and Parameters
# ============================================================
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from torch.utils.data import DataLoader, Dataset

# Root folder that contains shared bin-specific train/val cnn_dataset CSV files.
DATASET_DIR = Path(r"C:\Users\jianjing\Desktop\Fiseha\Ramp_Detection\training\latest dataset")
N_BINS = 192
MODEL_VERSION = "v2"
TRAIN_DIR = DATASET_DIR / f"train_{N_BINS}bin"
VAL_DIR = DATASET_DIR / f"val_{N_BINS}bin"
DATASET_GLOB = "*cnn_dataset*.csv"
# Folder where the trained PyTorch model and metadata will be saved.
OUTPUT_DIR = None  # Set after SPLIT_MODE so zone-wise CV can use a separate output folder.

# The model sees three aligned 1D channels per sample:
#   z   = profile elevation values
#   gap = bins filled by interpolation inside the observed extent
#   pad = bins that are just trailing zero-padding after last_valid_bin
CHANNELS = ["z", "gap", "pad"]
FEATURE_COLS = ["feat_kink_dz", "feat_kink_slope"]
USE_SCALAR_FEATURES = True
ACTIVE_FEATURE_COLS = FEATURE_COLS if USE_SCALAR_FEATURES else []
LABEL_MAP = {
    0: "RAMP",
    1: "CURB_NO_RAMP",
    4: "DEPRESSED_DITCH",
}
VALID_CLASS_IDS = sorted(LABEL_MAP.keys())
CLASS_TO_INDEX = {cls_id: idx for idx, cls_id in enumerate(VALID_CLASS_IDS)}
INDEX_TO_CLASS = {idx: cls_id for cls_id, idx in CLASS_TO_INDEX.items()}
INDEX_TO_LABEL = {idx: LABEL_MAP[cls_id] for cls_id, idx in CLASS_TO_INDEX.items()}

# Training hyperparameters.
BATCH_SIZE = 128
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 500
EARLY_STOPPING_PATIENCE = 100
DROPOUT = 0.1
RANDOM_SEED = 0
USE_Z_SCORE_ON_Z_CHANNEL = False
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 1

# Contextual station-embedding model settings.
USE_CONTEXT_MODEL = True
CONTEXT_WINDOW_SIZE = 11
CONTEXT_HALF_WIDTH = CONTEXT_WINDOW_SIZE // 2
STATION_EMBEDDING_DIM = 64
MAX_CONTEXT_GAP_M = 0.4
USE_SPATIAL_ATTENTION = False
# Optional ramp-only augmentation applied after the train/val split.
# Augmentation is intentionally minute so profile geometry stays realistic.
ENABLE_RAMP_AUG = False
RAMP_AUG_FACTOR = 1
RAMP_NOISE_STD = 0.0025
RAMP_OFFSET_RANGE = 0.005
PLOT_RAMP_AUG_SAMPLES = 5
# Optional per-class caps applied right after loading the combined dataset.
# Use None to keep all rows for a class. Example: {"DEPRESSED_DITCH": 1000}
MAX_SAMPLES_PER_LABEL = {
    "RAMP": None,
    "CURB_NO_RAMP": None,
    "DEPRESSED_DITCH": None,
}
# Split mode options:
#   folder_holdout    -> use TRAIN_DIR as train and VAL_DIR as validation directly
#   zone_holdout_cv   -> hold out one complete zone at a time, using all train/val files
SPLIT_MODE = "zone_holdout_cv"
# Zone-wise CV options. Uses zones parsed from names like cnn_dataset_zone03_p2.csv.
CV_HOLDOUT_ZONES = None  # None runs one fold per discovered zone; or set e.g. ["03", "11"].
CV_SELECTION_METRIC = "macro_f1_present_classes"
TRAIN_FINAL_MODEL_AFTER_CV = True
FINAL_EPOCH_REDUCTION = "median"

if SPLIT_MODE == "zone_holdout_cv":
    OUTPUT_DIR = DATASET_DIR / MODEL_VERSION / f"{N_BINS}bin_zone_cv_outputs"
else:
    OUTPUT_DIR = DATASET_DIR / MODEL_VERSION / f"{N_BINS}bin_outputs"

# Kept only for legacy/internal experiments; not used by the default v2 folder holdout.
VAL_SIZE = 0.2

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=RANDOM_SEED):
    """Seed Python, NumPy, and PyTorch for repeatable experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dataset dir : {DATASET_DIR}")
print(f"Train dir   : {TRAIN_DIR}")
print(f"Val dir     : {VAL_DIR}")
print(f"Dataset glob: {DATASET_GLOB}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Device     : {device}")
print(f"Classes    : {INDEX_TO_LABEL}")
print(f"Use scalar features: {USE_SCALAR_FEATURES}")
print(f"Feature cols: {ACTIVE_FEATURE_COLS}")
print(f"Use focal loss: {USE_FOCAL_LOSS} (gamma={FOCAL_GAMMA})")
print(f"Per-class caps: {MAX_SAMPLES_PER_LABEL}")
print(f"Ramp augmentation enabled: {ENABLE_RAMP_AUG} | factor={RAMP_AUG_FACTOR} | noise_std={RAMP_NOISE_STD} | offset_range={RAMP_OFFSET_RANGE}")
print(f"Split mode: {SPLIT_MODE}")
print(f"Context model: {USE_CONTEXT_MODEL} | window={CONTEXT_WINDOW_SIZE} | embedding_dim={STATION_EMBEDDING_DIM} | max_gap={MAX_CONTEXT_GAP_M}")


In [ ]:
# ============================================================
# CELL 1.5 - V2 Zone-Wise Cross-Validation and Final Model
# ============================================================
# Active only when SPLIT_MODE == "zone_holdout_cv". This runs one held-out-zone
# fold per discovered zone, rebuilds context windows inside each fold, aggregates
# all held-out predictions, then trains one final contextual model on all zones
# for the median best epoch from CV.

if SPLIT_MODE != "zone_holdout_cv":
    print("V2 zone-wise CV disabled; continuing with the standard training workflow.")
else:
    from ramp_zone_cv_training_v2 import run_zone_holdout_cv_v2
    zone_cv_results = run_zone_holdout_cv_v2(globals())


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 2 - Load Dataset and Build Arrays
    # ============================================================
    # Discover dataset CSV files from the configured train/val folders.
    train_dataset_files = sorted(TRAIN_DIR.glob(DATASET_GLOB))
    val_dataset_files = sorted(VAL_DIR.glob(DATASET_GLOB))

    if SPLIT_MODE != "folder_holdout":
        raise ValueError("V2 expects SPLIT_MODE='folder_holdout'.")
    if not train_dataset_files:
        raise FileNotFoundError(f"No dataset CSVs found in train dir {TRAIN_DIR} matching pattern {DATASET_GLOB!r}")
    if not val_dataset_files:
        raise FileNotFoundError(f"SPLIT_MODE='folder_holdout' requires validation CSVs in {VAL_DIR}, but none were found.")

    def load_dataset_frames(file_list):
        frames = []
        for csv_path in file_list:
            part = pd.read_csv(csv_path).copy()
            part = part.assign(dataset_csv_path=str(csv_path.resolve()), dataset_csv_name=csv_path.name)
            frames.append(part)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    train_df_raw = load_dataset_frames(train_dataset_files)
    val_df_raw = load_dataset_frames(val_dataset_files)

    z_cols = [f"z_{i:03d}" for i in range(N_BINS)]
    gap_cols = [f"gap_{i:03d}" for i in range(N_BINS)]
    pad_cols = [f"pad_{i:03d}" for i in range(N_BINS)]
    required_cols = z_cols + gap_cols + pad_cols + ACTIVE_FEATURE_COLS + ["class_int", "label", "side", "s_m", "source_file", "dataset_csv_path"]

    def check_required_columns(frame, frame_name):
        missing = [c for c in required_cols if c not in frame.columns]
        if missing:
            raise ValueError(f"Missing required columns in {frame_name}: {missing[:10]}{'...' if len(missing) > 10 else ''}")

    check_required_columns(train_df_raw, "train_df_raw")
    check_required_columns(val_df_raw, "val_df_raw")

    # Keep only the three target classes used for the ramp classifier.
    train_df_raw = train_df_raw[train_df_raw["class_int"].isin(VALID_CLASS_IDS)].copy()
    val_df_raw = val_df_raw[val_df_raw["class_int"].isin(VALID_CLASS_IDS)].copy()

    # Optional caps are applied only to training rows in folder-holdout mode.
    def apply_label_caps(frame, cap_map, seed):
        cap_records = []
        capped_frames = []
        for label_name, group in frame.groupby("label", sort=False):
            cap = cap_map.get(label_name, None)
            original_n = len(group)
            if cap is None or cap >= original_n:
                kept_group = group.copy()
                kept_n = original_n
            else:
                kept_group = group.sample(n=int(cap), random_state=seed).copy()
                kept_n = len(kept_group)
            capped_frames.append(kept_group)
            cap_records.append((label_name, original_n, kept_n, cap))
        out = pd.concat(capped_frames, ignore_index=True) if capped_frames else frame.copy()
        return out, cap_records

    train_df_raw, cap_records = apply_label_caps(train_df_raw, MAX_SAMPLES_PER_LABEL, RANDOM_SEED)

    if train_df_raw.empty or val_df_raw.empty:
        raise ValueError("After target-class filtering, both train and validation folders must contain at least one row.")

    train_df_raw["target_idx"] = train_df_raw["class_int"].map(CLASS_TO_INDEX).astype(int)
    val_df_raw["target_idx"] = val_df_raw["class_int"].map(CLASS_TO_INDEX).astype(int)
    df = pd.concat([train_df_raw, val_df_raw], ignore_index=True)

    # Precompute profile tensor for quick dataset-wide sanity checks.
    z = df[z_cols].to_numpy(dtype=np.float32)
    gap = df[gap_cols].to_numpy(dtype=np.float32)
    pad = df[pad_cols].to_numpy(dtype=np.float32)
    X = np.stack([z, gap, pad], axis=1)
    X_feat = df[ACTIVE_FEATURE_COLS].to_numpy(dtype=np.float32)
    y = df["target_idx"].to_numpy(dtype=np.int64)

    print(f"Loaded train dataset files: {len(train_dataset_files)}")
    for p in train_dataset_files:
        print(f"  train: {p.name}")
    print(f"Loaded val dataset files: {len(val_dataset_files)}")
    for p in val_dataset_files:
        print(f"  val  : {p.name}")
    print(f"Retained target-class rows: {len(df):,}")
    print(f"Train rows: {len(train_df_raw):,} | Val rows: {len(val_df_raw):,}")
    print(f"Profile tensor shape: {X.shape}")
    print(f"Feature tensor shape: {X_feat.shape}")
    print(f"Target shape: {y.shape}")
    print("Class distribution:")
    print(df["label"].value_counts().to_string())
    print()
    print("Cap summary:")
    for label_name, original_n, kept_n, cap in cap_records:
        print(f"  {label_name}: original={original_n:,} kept={kept_n:,} cap={cap}")
    print()
    print("Rows per source_file:")
    print(df["source_file"].value_counts().to_string())
    df[["label", "class_int", "s_m", "side", "source_file", "dataset_csv_name"]].head()


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 3 - Train/Val Split
    # ============================================================
    # V2 uses folder holdout so context windows are built only inside one split.
    # Put complete source segments/files in TRAIN_DIR or VAL_DIR before running.
    if USE_CONTEXT_MODEL and SPLIT_MODE != "folder_holdout":
        raise ValueError("The contextual model requires SPLIT_MODE='folder_holdout' to avoid neighbor leakage across splits.")

    split_records = []

    if SPLIT_MODE == "folder_holdout":
        train_df = train_df_raw.copy()
        val_df = val_df_raw.copy()
        for csv_name in train_df["dataset_csv_name"].drop_duplicates().tolist():
            total = int((train_df["dataset_csv_name"] == csv_name).sum())
            split_records.append((csv_name, total, total, 0, "folder_train"))
        for csv_name in val_df["dataset_csv_name"].drop_duplicates().tolist():
            total = int((val_df["dataset_csv_name"] == csv_name).sum())
            split_records.append((csv_name, total, 0, total, "folder_val"))
        split_mode = "train folder vs val folder holdout"
    else:
        raise ValueError(f"Unknown SPLIT_MODE={SPLIT_MODE!r}. V2 expects folder_holdout.")

    if train_df.empty or val_df.empty:
        raise ValueError("Folder holdout requires non-empty train and validation dataframes.")

    train_sources = set(train_df["source_file"].astype(str))
    val_sources = set(val_df["source_file"].astype(str))
    overlap_sources = sorted(train_sources & val_sources)
    if overlap_sources:
        raise ValueError(f"Source files appear in both train and validation splits: {overlap_sources[:5]}")

    print(f"Split mode: {split_mode}")
    print(f"Train: {len(train_df):,}")
    print(f"Val  : {len(val_df):,}")

    print()
    print("Split summary:")
    for source_name, source_size, n_train, n_val, mode in split_records:
        print(f"  {source_name} | total={source_size} train={n_train} val={n_val} mode={mode}")
    print()
    print("Train source files:")
    print(train_df["source_file"].drop_duplicates().to_list())
    print()
    print("Val source files:")
    print(val_df["source_file"].drop_duplicates().to_list())
    print()
    print("Train dataset files:")
    print(train_df["dataset_csv_name"].drop_duplicates().to_list())
    print()
    print("Val dataset files:")
    print(val_df["dataset_csv_name"].drop_duplicates().to_list())
    print()
    print("Train class distribution:")
    print(train_df["label"].value_counts().to_string())
    print()
    print("Val class distribution:")
    print(val_df["label"].value_counts().to_string())


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 3.5 - Finite-Safe Scalar Feature Imputation
    # ============================================================
    # Some manually corrected labels can have missing rule-based scalar metrics.
    # Fill those values from train-set finite means before augmentation/tensor conversion.
    train_df = train_df.copy()
    val_df = val_df.copy()
    feature_impute_values = {}
    if not ACTIVE_FEATURE_COLS:
        print("Scalar features disabled; skipping scalar feature imputation.")

    for col in ACTIVE_FEATURE_COLS:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        val_df[col] = pd.to_numeric(val_df[col], errors="coerce")

        train_values = train_df[col].to_numpy(dtype=np.float32)
        val_values = val_df[col].to_numpy(dtype=np.float32)
        train_finite = np.isfinite(train_values)
        val_finite = np.isfinite(val_values)

        impute_value = float(train_values[train_finite].mean()) if train_finite.any() else 0.0
        feature_impute_values[col] = impute_value

        missing_train = int((~train_finite).sum())
        missing_val = int((~val_finite).sum())
        if missing_train or missing_val:
            print(f"{col}: filled missing/non-finite values with train mean {impute_value:.6f} | train={missing_train} val={missing_val}")

        train_df.loc[~train_finite, col] = impute_value
        val_df.loc[~val_finite, col] = impute_value

    if ACTIVE_FEATURE_COLS:
        train_feat_check = train_df[ACTIVE_FEATURE_COLS].to_numpy(dtype=np.float32)
        val_feat_check = val_df[ACTIVE_FEATURE_COLS].to_numpy(dtype=np.float32)
        if not np.isfinite(train_feat_check).all() or not np.isfinite(val_feat_check).all():
            raise ValueError("Non-finite scalar features remain after imputation.")

    print(f"Scalar feature imputation values: {feature_impute_values}")


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 4 - Ramp Augmentation (Train Only)
    # ============================================================
    # Validation remains untouched. Only training RAMP rows are duplicated with tiny perturbations.
    # V2 context windows require ordered station rows, so row-level augmentation stays disabled.
    if USE_CONTEXT_MODEL and ENABLE_RAMP_AUG:
        raise ValueError("Disable ENABLE_RAMP_AUG for contextual training; use sequence-level augmentation later if needed.")
    train_df = train_df.copy().reset_index(drop=True)
    val_df = val_df.copy().reset_index(drop=True)
    train_df["is_augmented"] = 0
    val_df["is_augmented"] = 0
    train_df["aug_parent_id"] = np.arange(len(train_df))
    val_df["aug_parent_id"] = -1

    def augment_ramp_rows(train_df, z_cols, random_seed=RANDOM_SEED):
        """Create minute augmented copies of training RAMP rows.

        Only z_* values inside the valid region are perturbed. gap_* and pad_* are left unchanged.
        """
        if (not ENABLE_RAMP_AUG) or RAMP_AUG_FACTOR <= 0:
            return train_df.copy(), pd.DataFrame()

        rng = np.random.default_rng(random_seed)
        ramp_df = train_df[train_df["label"] == "RAMP"].copy()
        aug_rows = []

        for _, row in ramp_df.iterrows():
            parent_id = int(row["aug_parent_id"])
            last_valid_bin = int(row["last_valid_bin"])
            valid_len = last_valid_bin + 1
            if valid_len <= 0:
                continue

            base_z = row[z_cols].to_numpy(dtype=float)
            for aug_id in range(RAMP_AUG_FACTOR):
                z_aug = base_z.copy()
                offset = rng.uniform(-RAMP_OFFSET_RANGE, RAMP_OFFSET_RANGE)
                noise = rng.normal(0.0, RAMP_NOISE_STD, size=valid_len)
                z_aug[:valid_len] = z_aug[:valid_len] + offset + noise
                z_aug[valid_len:] = 0.0

                new_row = row.copy()
                for i, col in enumerate(z_cols):
                    new_row[col] = float(z_aug[i])
                new_row["is_augmented"] = 1
                new_row["aug_parent_id"] = parent_id
                aug_rows.append(new_row)

        aug_df = pd.DataFrame(aug_rows)
        if aug_df.empty:
            return train_df.copy(), aug_df

        train_df_final = pd.concat([train_df, aug_df], ignore_index=True)
        train_df_final = train_df_final.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)
        return train_df_final, aug_df

    train_df_final, ramp_aug_df = augment_ramp_rows(train_df, z_cols)
    val_df_final = val_df.copy()

    print(f"Original train rows: {len(train_df):,}")
    print(f"Original train RAMP rows: {(train_df['label'] == 'RAMP').sum():,}")
    print(f"Augmented RAMP rows added: {len(ramp_aug_df):,}")
    print(f"Final train rows: {len(train_df_final):,}")
    print()
    print("Final train class distribution:")
    print(train_df_final['label'].value_counts().to_string())

    v_axis = np.arange(N_BINS) * 0.08
    if ENABLE_RAMP_AUG and not ramp_aug_df.empty and PLOT_RAMP_AUG_SAMPLES > 0:
        plot_parents = ramp_aug_df['aug_parent_id'].drop_duplicates().to_numpy()[:PLOT_RAMP_AUG_SAMPLES]
        n_plot = len(plot_parents)
        fig, axes = plt.subplots(n_plot, 1, figsize=(12, 3.5 * n_plot), sharex=True)
        if n_plot == 1:
            axes = [axes]

        for ax, parent_id in zip(axes, plot_parents):
            orig_row = train_df[train_df['aug_parent_id'] == parent_id].iloc[0]
            aug_row = ramp_aug_df[ramp_aug_df['aug_parent_id'] == parent_id].iloc[0]
            orig_z = orig_row[z_cols].to_numpy(dtype=float)
            aug_z = aug_row[z_cols].to_numpy(dtype=float)
            last_valid_bin = int(orig_row['last_valid_bin'])
            valid_end_v = (last_valid_bin + 1) * 0.08 if last_valid_bin >= 0 else 0.0
            z_orig_plot = np.full(N_BINS, np.nan, dtype=float)
            z_aug_plot = np.full(N_BINS, np.nan, dtype=float)
            if last_valid_bin >= 0:
                z_orig_plot[: last_valid_bin + 1] = orig_z[: last_valid_bin + 1]
                z_aug_plot[: last_valid_bin + 1] = aug_z[: last_valid_bin + 1]
                ax.axvspan(0.0, valid_end_v, color='orange', alpha=0.10, label='valid extent')
            ax.plot(v_axis, z_orig_plot, lw=1.8, label='original ramp')
            ax.plot(v_axis, z_aug_plot, lw=1.4, ls='--', label='augmented ramp')
            ax.set_title(f"RAMP augmentation | parent_id={parent_id} | s_m={orig_row['s_m']:.3f} | side={orig_row['side']}")
            ax.set_ylabel('z (m)')
            ax.grid(True)
            ax.legend(fontsize=8)

        axes[-1].set_xlabel('v (m)')
        plt.tight_layout()
        plt.show()


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 4 - Context Dataset, Normalization, and DataLoaders
    # ============================================================
    def df_to_arrays(frame, z_cols, gap_cols, pad_cols, feature_cols):
        """Convert rows into local profile tensors, scalar features, and compact targets."""
        z_arr = frame[z_cols].to_numpy(dtype=np.float32)
        gap_arr = frame[gap_cols].to_numpy(dtype=np.float32)
        pad_arr = frame[pad_cols].to_numpy(dtype=np.float32)
        X_arr = np.stack([z_arr, gap_arr, pad_arr], axis=1)
        X_feat_arr = frame[feature_cols].to_numpy(dtype=np.float32)
        y_arr = frame["target_idx"].to_numpy(dtype=np.int64)
        return X_arr, X_feat_arr, y_arr

    X_train_all, X_train_feat_all, y_train_all = df_to_arrays(train_df_final, z_cols, gap_cols, pad_cols, ACTIVE_FEATURE_COLS)
    X_val_all, X_val_feat_all, y_val_all = df_to_arrays(val_df_final, z_cols, gap_cols, pad_cols, ACTIVE_FEATURE_COLS)

    if USE_Z_SCORE_ON_Z_CHANNEL:
        z_train = X_train_all[:, 0, :]
        z_mean = float(z_train.mean())
        z_std = float(z_train.std())
        if z_std < 1e-8:
            z_std = 1.0
    else:
        z_mean = 0.0
        z_std = 1.0

    if USE_SCALAR_FEATURES:
        feat_mean = X_train_feat_all.mean(axis=0).astype(np.float32)
        feat_std = X_train_feat_all.std(axis=0).astype(np.float32)
        feat_std[feat_std < 1e-8] = 1.0
    else:
        feat_mean = np.zeros(0, dtype=np.float32)
        feat_std = np.ones(0, dtype=np.float32)

    class_counts = np.bincount(y_train_all, minlength=len(VALID_CLASS_IDS))
    class_weights = class_counts.sum() / np.maximum(class_counts, 1)
    class_weights = class_weights / class_weights.mean()
    class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)

    class ContextRampDataset(Dataset):
        """Build fixed station-context windows inside each source_file and side."""
        def __init__(self, frame, z_cols, gap_cols, pad_cols, feature_cols, z_mean, z_std, feat_mean, feat_std, context_window_size, max_context_gap_m):
            if context_window_size % 2 != 1:
                raise ValueError("context_window_size must be odd so there is one centre station.")
            self.frame = frame.copy().reset_index(drop=True)
            self.context_window_size = int(context_window_size)
            self.half_width = self.context_window_size // 2
            self.max_context_gap_m = max_context_gap_m

            X, X_feat, y = df_to_arrays(self.frame, z_cols, gap_cols, pad_cols, feature_cols)
            self.X = X.copy()
            self.X[:, 0, :] = (self.X[:, 0, :] - z_mean) / z_std
            self.X_feat = X_feat.copy()
            self.X_feat = (self.X_feat - feat_mean) / feat_std
            self.y = y.copy()

            self.samples = []
            self.center_indices = []
            self._build_windows()
            self.center_frame = self.frame.loc[self.center_indices].reset_index(drop=True)

        def _valid_neighbor(self, s_values, center_pos, neighbor_pos):
            if neighbor_pos < 0 or neighbor_pos >= len(s_values):
                return False
            if self.max_context_gap_m is None:
                return True
            lo = min(center_pos, neighbor_pos)
            hi = max(center_pos, neighbor_pos)
            if lo == hi:
                return True
            gaps = np.diff(s_values[lo: hi + 1])
            return bool(np.isfinite(gaps).all() and (gaps <= self.max_context_gap_m + 1e-6).all())

        def _build_windows(self):
            for _, group in self.frame.groupby(["source_file", "side"], sort=False):
                group_sorted = group.sort_values("s_m", kind="mergesort")
                group_indices = group_sorted.index.to_numpy(dtype=int)
                s_values = group_sorted["s_m"].to_numpy(dtype=float)
                for center_pos, center_idx in enumerate(group_indices):
                    window_indices = []
                    mask = []
                    for offset in range(-self.half_width, self.half_width + 1):
                        neighbor_pos = center_pos + offset
                        if self._valid_neighbor(s_values, center_pos, neighbor_pos):
                            window_indices.append(int(group_indices[neighbor_pos]))
                            mask.append(1.0)
                        else:
                            window_indices.append(-1)
                            mask.append(0.0)
                    window_indices[self.half_width] = int(center_idx)
                    mask[self.half_width] = 1.0
                    self.samples.append((window_indices, np.asarray(mask, dtype=np.float32), int(center_idx)))
                    self.center_indices.append(int(center_idx))

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            window_indices, mask, center_idx = self.samples[idx]
            x_seq = np.zeros((self.context_window_size, self.X.shape[1], self.X.shape[2]), dtype=np.float32)
            feat_seq = np.zeros((self.context_window_size, self.X_feat.shape[1]), dtype=np.float32)
            for pos, row_idx in enumerate(window_indices):
                if row_idx >= 0:
                    x_seq[pos] = self.X[row_idx]
                    feat_seq[pos] = self.X_feat[row_idx]
            return (
                torch.tensor(x_seq, dtype=torch.float32),
                torch.tensor(feat_seq, dtype=torch.float32),
                torch.tensor(mask, dtype=torch.float32),
                torch.tensor(self.y[center_idx], dtype=torch.long),
            )

    train_ds = ContextRampDataset(train_df_final, z_cols, gap_cols, pad_cols, ACTIVE_FEATURE_COLS, z_mean, z_std, feat_mean, feat_std, CONTEXT_WINDOW_SIZE, MAX_CONTEXT_GAP_M)
    val_ds = ContextRampDataset(val_df_final, z_cols, gap_cols, pad_cols, ACTIVE_FEATURE_COLS, z_mean, z_std, feat_mean, feat_std, CONTEXT_WINDOW_SIZE, MAX_CONTEXT_GAP_M)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(RANDOM_SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    print(f"z_mean={z_mean:.6f}, z_std={z_std:.6f}")
    print(f"feat_mean={feat_mean}")
    print(f"feat_std={feat_std}")
    print(f"Class weights: {class_weights}")
    print(f"Context window: T={CONTEXT_WINDOW_SIZE}, half_width={CONTEXT_HALF_WIDTH}, max_gap={MAX_CONTEXT_GAP_M}")
    print(f"Train context samples: {len(train_ds):,} | Val context samples: {len(val_ds):,}")
    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 5 - Contextual PyTorch 1D CNN Model
    # ============================================================
    class SpatialAttention1d(nn.Module):
        """Learns a soft weight over the length dimension."""
        def __init__(self, in_ch, reduction=8):
            super().__init__()
            self.attn = nn.Sequential(
                nn.Conv1d(in_ch, in_ch // reduction, kernel_size=1),
                nn.LeakyReLU(0.1),
                nn.Conv1d(in_ch // reduction, 1, kernel_size=1),
                nn.Sigmoid(),
            )

        def forward(self, x):
            return x * self.attn(x)

    class FocalLoss(nn.Module):
        """Multiclass focal loss for imbalanced classification."""
        def __init__(self, gamma=2.0, weight=None, reduction="mean"):
            super().__init__()
            self.gamma = gamma
            self.weight = weight
            self.reduction = reduction

        def forward(self, logits, targets):
            ce = nn.functional.cross_entropy(logits, targets, weight=self.weight, reduction="none")
            pt = torch.exp(-ce)
            loss = ((1.0 - pt) ** self.gamma) * ce
            if self.reduction == "mean":
                return loss.mean()
            if self.reduction == "sum":
                return loss.sum()
            return loss

    class StationEncoder(nn.Module):
        """Shared local encoder for one station-side lateral profile."""
        def __init__(self, in_channels=3, num_features=2, embedding_dim=64, use_spatial_attention=False, dropout=DROPOUT):
            super().__init__()
            self.use_scalar_features = num_features > 0
            self.use_spatial_attention = use_spatial_attention
            self.features = nn.Sequential(
                nn.Conv1d(in_channels, 64, kernel_size=7, padding="same"), nn.BatchNorm1d(64), nn.LeakyReLU(0.1), nn.MaxPool1d(2),
                nn.Conv1d(64, 128, kernel_size=5, padding="same"), nn.BatchNorm1d(128), nn.LeakyReLU(0.1), nn.MaxPool1d(2),
                nn.Conv1d(128, 128, kernel_size=3, padding="same"), nn.BatchNorm1d(128), nn.LeakyReLU(0.1),
            )
            self.attn = SpatialAttention1d(128, reduction=8)
            self.pool = nn.AdaptiveAvgPool1d(1)
            self.profile_head = nn.Sequential(nn.Flatten(), nn.Linear(128, 64), nn.LeakyReLU(0.1))
            if self.use_scalar_features:
                self.feature_head = nn.Sequential(nn.Linear(num_features, 16), nn.LeakyReLU(0.1))
                local_dim = 64 + 16
            else:
                self.feature_head = None
                local_dim = 64
            self.station_embedding = nn.Sequential(nn.Linear(local_dim, embedding_dim), nn.LeakyReLU(0.1), nn.Dropout(dropout))

        def forward(self, x, x_feat):
            x = self.features(x)
            if self.use_spatial_attention:
                x = self.attn(x)
            x = self.pool(x)
            x = self.profile_head(x)
            if self.use_scalar_features:
                x_feat = self.feature_head(x_feat)
                x = torch.cat([x, x_feat], dim=1)
            return self.station_embedding(x)

    class RampCNN(nn.Module):
        """Contextual ramp classifier with station-embedding fusion."""
        def __init__(self, in_channels=3, num_features=2, num_classes=3, embedding_dim=64, dropout=DROPOUT):
            super().__init__()
            self.embedding_dim = embedding_dim
            self.station_encoder = StationEncoder(in_channels, num_features, embedding_dim, USE_SPATIAL_ATTENTION, dropout)
            self.context_encoder = nn.Sequential(
                nn.Conv1d(embedding_dim + 1, embedding_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embedding_dim), nn.LeakyReLU(0.1), nn.Dropout(dropout),
                nn.Conv1d(embedding_dim, embedding_dim, kernel_size=3, padding=1), nn.BatchNorm1d(embedding_dim), nn.LeakyReLU(0.1), nn.Dropout(dropout),
            )
            self.context_norm = nn.LayerNorm(embedding_dim)
            self.classifier = nn.Sequential(nn.Linear(embedding_dim, 64), nn.LeakyReLU(0.1), nn.Dropout(dropout), nn.Linear(64, num_classes))

        def forward(self, x_sequence, feature_sequence, context_mask):
            B, T, C, L = x_sequence.shape
            x_flat = x_sequence.reshape(B * T, C, L)
            feature_flat = feature_sequence.reshape(B * T, -1)
            embedding_flat = self.station_encoder(x_flat, feature_flat)
            embedding_sequence = embedding_flat.reshape(B, T, self.embedding_dim)
            mask = context_mask.unsqueeze(-1)
            embedding_masked = embedding_sequence * mask
            context_input = torch.cat([embedding_masked, mask], dim=-1).transpose(1, 2)
            contextual_embeddings = self.context_encoder(context_input).transpose(1, 2) * mask
            combined_embeddings = self.context_norm(embedding_masked + contextual_embeddings)
            center_embedding = combined_embeddings[:, T // 2, :]
            return self.classifier(center_embedding)

    model = RampCNN(len(CHANNELS), len(ACTIVE_FEATURE_COLS), len(VALID_CLASS_IDS), STATION_EMBEDDING_DIM).to(device)
    if USE_FOCAL_LOSS:
        criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights_t)
        print(f"Using focal loss with gamma={FOCAL_GAMMA}")
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights_t)
        print("Using weighted cross-entropy loss")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
    print(model)


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 6 - Train the Contextual Model
    # ============================================================
    def run_epoch(model, loader, criterion, optimizer=None):
        """Run one full pass over a contextual dataloader."""
        train_mode = optimizer is not None
        model.train(train_mode)
        total_loss = 0.0
        total_correct = 0
        total_count = 0
        all_preds = []
        all_targets = []

        for xb, xb_feat, xb_mask, yb in loader:
            xb = xb.to(device)
            xb_feat = xb_feat.to(device)
            xb_mask = xb_mask.to(device)
            yb = yb.to(device)
            with torch.set_grad_enabled(train_mode):
                logits = model(xb, xb_feat, xb_mask)
                loss = criterion(logits, yb)
                if train_mode:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
            total_loss += float(loss.item()) * xb.size(0)
            preds = logits.argmax(dim=1)
            total_correct += int((preds == yb).sum().item())
            total_count += int(xb.size(0))
            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(yb.detach().cpu().numpy())

        y_pred = np.concatenate(all_preds, axis=0)
        y_true = np.concatenate(all_targets, axis=0)
        macro_f1 = f1_score(y_true, y_pred, labels=list(range(len(VALID_CLASS_IDS))), average="macro", zero_division=0)
        return total_loss / total_count, total_correct / total_count, macro_f1

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "train_f1": [], "val_f1": []}
    best_val_loss = float("inf")
    best_val_f1 = -float("inf")
    best_state = None
    best_epoch = -1
    epochs_without_improve = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc, train_f1 = run_epoch(model, train_loader, criterion, optimizer=optimizer)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, criterion, optimizer=None)
        scheduler.step(val_loss)
        history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc); history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1); history["val_f1"].append(val_f1)
        print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_f1={train_f1:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch}. Best epoch was {best_epoch}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best model from epoch {best_epoch} with val_f1={best_val_f1:.4f} and val_loss={best_val_loss:.4f}")


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 7 - Evaluate on Validation Set
    # ============================================================
    model.eval()
    all_logits = []
    all_targets = []
    with torch.no_grad():
        for xb, xb_feat, xb_mask, yb in val_loader:
            logits = model(xb.to(device), xb_feat.to(device), xb_mask.to(device))
            all_logits.append(logits.cpu().numpy())
            all_targets.append(yb.numpy())

    logits = np.concatenate(all_logits, axis=0)
    y_true = np.concatenate(all_targets, axis=0)
    y_pred = logits.argmax(axis=1)
    eval_labels = list(range(len(INDEX_TO_LABEL)))
    label_names = [INDEX_TO_LABEL[i] for i in eval_labels]
    display_label_names = [name.replace("_", "\n") for name in label_names]
    val_eval_acc = (y_pred == y_true).mean()
    val_eval_f1 = f1_score(y_true, y_pred, labels=eval_labels, average="macro", zero_division=0)
    print(f"Validation accuracy: {val_eval_acc:.4f}")
    print(f"Validation macro F1: {val_eval_f1:.4f}")
    print()
    print(classification_report(y_true, y_pred, labels=eval_labels, target_names=label_names, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=eval_labels)
    cm_norm = confusion_matrix(y_true, y_pred, labels=eval_labels, normalize="true")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=display_label_names, yticklabels=display_label_names, ax=axes[0])
    axes[0].set_title("Confusion Matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].tick_params(axis="x", rotation=0, labelsize=10); axes[0].tick_params(axis="y", rotation=0, labelsize=10)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=display_label_names, yticklabels=display_label_names, ax=axes[1])
    axes[1].set_title("Normalized Confusion Matrix"); axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
    axes[1].tick_params(axis="x", rotation=0, labelsize=10); axes[1].tick_params(axis="y", rotation=0, labelsize=10)
    plt.tight_layout(pad=2.0)
    confusion_matrix_path = OUTPUT_DIR / "confusion_matrix.png"
    fig.savefig(confusion_matrix_path, dpi=200, bbox_inches="tight")
    print(f"Saved confusion matrix -> {confusion_matrix_path}")
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(history["train_loss"], label="Train Loss"); axes[0].plot(history["val_loss"], label="Val Loss"); axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(history["train_acc"], label="Train Acc"); axes[1].plot(history["val_acc"], label="Val Acc"); axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].grid(True)
    axes[2].plot(history["train_f1"], label="Train Macro F1"); axes[2].plot(history["val_f1"], label="Val Macro F1"); axes[2].set_title("Macro F1"); axes[2].legend(); axes[2].grid(True)
    plt.tight_layout()
    training_curves_path = OUTPUT_DIR / "training_curves.png"
    fig.savefig(training_curves_path, dpi=200, bbox_inches="tight")
    print(f"Saved training curves -> {training_curves_path}")
    plt.show()

    fig_loss, ax_loss = plt.subplots(figsize=(8, 5))
    ax_loss.plot(history["train_loss"], label="Train Loss")
    ax_loss.plot(history["val_loss"], label="Val Loss")
    ax_loss.set_title("Loss")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()
    ax_loss.grid(True)
    plt.tight_layout()
    loss_curve_path = OUTPUT_DIR / "loss_curve.png"
    fig_loss.savefig(loss_curve_path, dpi=200, bbox_inches="tight")
    print(f"Saved loss curve -> {loss_curve_path}")
    plt.show()

    history_df = pd.DataFrame(history)
    history_df.index = np.arange(1, len(history_df) + 1)
    history_df.index.name = "epoch"
    history_path = OUTPUT_DIR / "training_history.csv"
    history_df.to_csv(history_path)
    print(f"Saved training history -> {history_path}")


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 7.5 - Per-File Train/Validation Metrics
    # ============================================================
    def predict_dataframe(frame, split_name):
        """Run the selected contextual model and return centre-row predictions."""
        eval_ds = ContextRampDataset(frame, z_cols, gap_cols, pad_cols, ACTIVE_FEATURE_COLS, z_mean, z_std, feat_mean, feat_std, CONTEXT_WINDOW_SIZE, MAX_CONTEXT_GAP_M)
        eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)
        model.eval()
        logits_parts = []
        with torch.no_grad():
            for xb, xb_feat, xb_mask, _ in eval_loader:
                logits_parts.append(model(xb.to(device), xb_feat.to(device), xb_mask.to(device)).cpu().numpy())
        logits_eval = np.concatenate(logits_parts, axis=0)
        probs_eval = torch.softmax(torch.tensor(logits_eval), dim=1).numpy()
        pred_idx = probs_eval.argmax(axis=1)
        y_eval = eval_ds.y[eval_ds.center_indices]
        out = eval_ds.center_frame.copy().reset_index(drop=True)
        out["split"] = split_name
        out["true_idx"] = y_eval
        out["pred_idx"] = pred_idx
        out["true_label"] = out["true_idx"].map(INDEX_TO_LABEL)
        out["pred_label"] = out["pred_idx"].map(INDEX_TO_LABEL)
        out["pred_confidence"] = probs_eval.max(axis=1)
        out["correct"] = out["pred_idx"] == out["true_idx"]
        for class_idx, label_name in INDEX_TO_LABEL.items():
            out[f"prob_{label_name}"] = probs_eval[:, class_idx]
        return out

    def summarize_per_file(pred_df):
        """Build one metrics row per split/file."""
        rows = []
        for (split_name, csv_name), group in pred_df.groupby(["split", "dataset_csv_name"], sort=True):
            y_true_file = group["true_idx"].to_numpy(dtype=int)
            y_pred_file = group["pred_idx"].to_numpy(dtype=int)
            present_labels = sorted(np.unique(y_true_file).tolist())
            macro_f1_all_classes = f1_score(y_true_file, y_pred_file, labels=eval_labels, average="macro", zero_division=0)
            macro_f1_present_classes = f1_score(y_true_file, y_pred_file, labels=present_labels, average="macro", zero_division=0)
            row = {"split": split_name, "dataset_csv_name": csv_name, "n_samples": int(len(group)), "accuracy": float((y_true_file == y_pred_file).mean()) if len(group) else np.nan, "macro_f1": float(macro_f1_present_classes), "macro_f1_present_classes": float(macro_f1_present_classes), "macro_f1_all_classes": float(macro_f1_all_classes), "present_classes": ",".join(INDEX_TO_LABEL[i] for i in present_labels)}
            for class_idx, label_name in INDEX_TO_LABEL.items():
                true_mask = y_true_file == class_idx
                pred_mask = y_pred_file == class_idx
                row[f"n_{label_name}"] = int(true_mask.sum())
                row[f"correct_{label_name}"] = int((true_mask & pred_mask).sum())
                row[f"recall_{label_name}"] = float((true_mask & pred_mask).sum() / true_mask.sum()) if true_mask.any() else np.nan
                row[f"pred_{label_name}"] = int(pred_mask.sum())
            rows.append(row)
        return pd.DataFrame(rows)

    train_pred_by_row = predict_dataframe(train_df_final, "train")
    val_pred_by_row = predict_dataframe(val_df_final, "val")
    per_file_metrics_train = summarize_per_file(train_pred_by_row)
    per_file_metrics_val = summarize_per_file(val_pred_by_row)
    per_file_metrics_all = pd.concat([per_file_metrics_train, per_file_metrics_val], ignore_index=True)
    print("Train per-file metrics:"); display(per_file_metrics_train.sort_values(["macro_f1_present_classes", "accuracy", "n_samples"], ascending=[True, True, False]))
    print("Validation per-file metrics:"); display(per_file_metrics_val.sort_values(["macro_f1_present_classes", "accuracy", "n_samples"], ascending=[True, True, False]))

    row_output_cols = ["split", "dataset_csv_name", "source_file", "s_m", "side", "true_label", "pred_label", "pred_confidence", "correct", "class_int", "target_idx", "pred_idx", "is_augmented", "aug_parent_id"] + [f"prob_{label_name}" for label_name in INDEX_TO_LABEL.values()]
    row_output_cols = [c for c in row_output_cols if c in train_pred_by_row.columns or c in val_pred_by_row.columns]
    per_file_train_path = OUTPUT_DIR / "per_file_metrics_train.csv"
    per_file_val_path = OUTPUT_DIR / "per_file_metrics_val.csv"
    per_file_all_path = OUTPUT_DIR / "per_file_metrics_all.csv"
    train_rows_path = OUTPUT_DIR / "train_predictions_by_row.csv"
    val_rows_path = OUTPUT_DIR / "val_predictions_by_row.csv"
    per_file_metrics_train.to_csv(per_file_train_path, index=False)
    per_file_metrics_val.to_csv(per_file_val_path, index=False)
    per_file_metrics_all.to_csv(per_file_all_path, index=False)
    train_pred_by_row[row_output_cols].to_csv(train_rows_path, index=False)
    val_pred_by_row[row_output_cols].to_csv(val_rows_path, index=False)
    print(f"Saved train per-file metrics -> {per_file_train_path}")
    print(f"Saved val per-file metrics   -> {per_file_val_path}")
    print(f"Saved all per-file metrics   -> {per_file_all_path}")
    print(f"Saved train row predictions  -> {train_rows_path}")
    print(f"Saved val row predictions    -> {val_rows_path}")


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 7b - Inspect Misclassified Centre Stations For One GT Class
    # ============================================================
    GT_LABEL_TO_INSPECT = "CURB_NO_RAMP"
    GT_CLASS_INT = next(k for k, v in LABEL_MAP.items() if v == GT_LABEL_TO_INSPECT)
    GT_IDX = CLASS_TO_INDEX[GT_CLASS_INT]
    MAX_MISCLASSIFIED_PLOTS = 20
    PLOTS_PER_FIGURE = 4
    gt_mask = y_true == GT_IDX
    wrong_mask = y_pred != GT_IDX
    miss_idx = np.where(gt_mask & wrong_mask)[0]
    print(f"GT={GT_LABEL_TO_INSPECT} in val     : {gt_mask.sum():,}")
    print(f"Misclassified {GT_LABEL_TO_INSPECT} : {len(miss_idx):,}")

    if len(miss_idx) == 0:
        print(f"No misclassified {GT_LABEL_TO_INSPECT} samples - nothing to plot.")
    else:
        val_df_reset = val_ds.center_frame.reset_index(drop=True)
        v_axis = np.arange(N_BINS) * 0.08
        center_pos = CONTEXT_WINDOW_SIZE // 2
        n_plot = min(len(miss_idx), MAX_MISCLASSIFIED_PLOTS)
        plot_indices = miss_idx[:n_plot]
        for page_start in range(0, n_plot, PLOTS_PER_FIGURE):
            page_indices = plot_indices[page_start: page_start + PLOTS_PER_FIGURE]
            n_rows = len(page_indices)
            fig, axes = plt.subplots(n_rows, 1, figsize=(13, 4.5 * n_rows), sharex=True)
            if n_rows == 1:
                axes = [axes]
            for ax, vi in zip(axes, page_indices):
                x_seq_sample, x_feat_seq_sample, mask_sample, _ = val_ds[vi]
                x_sample = x_seq_sample[center_pos]
                x_feat_sample = x_feat_seq_sample[center_pos]
                z_norm = x_sample[0].numpy()
                gap_ch = x_sample[1].numpy().astype(bool)
                pad_ch = x_sample[2].numpy().astype(bool)
                feat_norm = x_feat_sample.numpy()
                row = val_df_reset.iloc[vi]
                pred_label = INDEX_TO_LABEL[int(y_pred[vi])]
                s_val = row.get("s_m", float("nan"))
                side_val = row.get("side", "?")
                src_val = row.get("source_file", "?")
                last_valid_bin = int(row.get("last_valid_bin", -1))
                feat_raw = row[ACTIVE_FEATURE_COLS].to_numpy(dtype=float) if ACTIVE_FEATURE_COLS else np.array([], dtype=float)
                valid_context_count = int(mask_sample.sum().item())
                if last_valid_bin >= 0:
                    valid_slice = slice(0, last_valid_bin + 1)
                    z_valid = z_norm[valid_slice].copy()
                    z_center = z_valid - np.nanmedian(z_valid)
                    z_plot = np.full(N_BINS, np.nan, dtype=float)
                    z_plot[valid_slice] = z_center
                    y_min = float(np.nanmin(z_center)); y_max = float(np.nanmax(z_center)); y_pad = max(0.01, 0.15 * max(y_max - y_min, 1e-6))
                    ax.axvspan(0.0, v_axis[last_valid_bin], color="orange", alpha=0.08, label="valid extent")
                    ax.plot(v_axis, z_plot, lw=2.0, color="tab:red", label="z (locally centered)")
                    ax.axhline(0.0, color="gray", lw=0.8, ls=":")
                    ax.axvline(v_axis[last_valid_bin], color="red", lw=1.0, ls="--", label="last valid bin")
                    gap_regions = gap_ch & (~pad_ch)
                    if gap_regions.any():
                        ax.fill_between(v_axis, y_min - y_pad, y_max + y_pad, where=gap_regions, color="gold", alpha=0.25, label="gap bins")
                    ax.set_ylim(y_min - y_pad, y_max + y_pad)
                else:
                    ax.text(0.5, 0.5, "No valid bins", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(f"val[{vi}] GT={GT_LABEL_TO_INSPECT} -> pred={pred_label} | s={s_val:.2f} m | side={side_val} | context={valid_context_count}/{CONTEXT_WINDOW_SIZE} | {src_val}", fontsize=10)
                if ACTIVE_FEATURE_COLS:
                    feat_lines = ["raw feat: " + ", ".join(f"{name}={val:.4f}" for name, val in zip(ACTIVE_FEATURE_COLS, feat_raw)), "norm feat: " + ", ".join(f"{name}={val:.4f}" for name, val in zip(ACTIVE_FEATURE_COLS, feat_norm))]
                else:
                    feat_lines = ["scalar features disabled"]
                feat_text = "\n".join(feat_lines + [f"station: s={s_val:.3f}, side={side_val}, last_valid_bin={last_valid_bin}"])
                ax.text(0.01, 0.98, feat_text, transform=ax.transAxes, va="top", ha="left", fontsize=8, bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.75, edgecolor="none"))
                ax.set_ylabel("relative z", fontsize=9); ax.tick_params(labelsize=8); ax.grid(True, linewidth=0.5); ax.legend(fontsize=8, loc="upper right")
            axes[-1].set_xlabel("v (m)  [lateral distance]", fontsize=9)
            page_end = page_start + n_rows
            fig.suptitle(f"Misclassified {GT_LABEL_TO_INSPECT} ({len(miss_idx)} total) - samples {page_start + 1}-{page_end}", fontsize=11)
            plt.tight_layout(); plt.show()


In [ ]:
if SPLIT_MODE == "zone_holdout_cv":
    print("Skipping standard workflow cell; V2 zone-wise CV ran in Cell 1.5.")
else:
    # ============================================================
    # CELL 8 - Save Model and Metadata
    # ============================================================
    model_path = OUTPUT_DIR / "ramp_cnn_pytorch.pt"
    meta_path = OUTPUT_DIR / "ramp_cnn_pytorch_meta.json"
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "model_type": "contextual_station_embedding_cnn",
        "model_version": MODEL_VERSION,
        "class_to_index": CLASS_TO_INDEX,
        "index_to_label": INDEX_TO_LABEL,
        "channels": CHANNELS,
        "use_scalar_features": USE_SCALAR_FEATURES,
        "feature_cols": ACTIVE_FEATURE_COLS,
        "available_feature_cols": FEATURE_COLS,
        "n_bins": N_BINS,
        "context_window_size": CONTEXT_WINDOW_SIZE,
        "context_half_width": CONTEXT_HALF_WIDTH,
        "station_embedding_dim": STATION_EMBEDDING_DIM,
        "max_context_gap_m": MAX_CONTEXT_GAP_M,
        "use_spatial_attention": USE_SPATIAL_ATTENTION,
    }
    torch.save(checkpoint, model_path)
    meta = {
        "model_type": "contextual_station_embedding_cnn",
        "model_version": MODEL_VERSION,
        "dataset_dir": str(DATASET_DIR),
        "train_dir": str(TRAIN_DIR),
        "val_dir": str(VAL_DIR),
        "dataset_glob": DATASET_GLOB,
        "train_dataset_files": [str(p.resolve()) for p in train_dataset_files],
        "val_dataset_files": [str(p.resolve()) for p in val_dataset_files],
        "n_train_dataset_files": len(train_dataset_files),
        "n_val_dataset_files": len(val_dataset_files),
        "output_dir": str(OUTPUT_DIR),
        "split_mode": SPLIT_MODE,
        "channels": CHANNELS,
        "use_scalar_features": USE_SCALAR_FEATURES,
        "feature_cols": ACTIVE_FEATURE_COLS,
        "available_feature_cols": FEATURE_COLS,
        "n_bins": N_BINS,
        "context_window_size": CONTEXT_WINDOW_SIZE,
        "context_half_width": CONTEXT_HALF_WIDTH,
        "station_embedding_dim": STATION_EMBEDDING_DIM,
        "max_context_gap_m": MAX_CONTEXT_GAP_M,
        "use_spatial_attention": USE_SPATIAL_ATTENTION,
        "class_to_index": CLASS_TO_INDEX,
        "index_to_label": INDEX_TO_LABEL,
        "z_mean": z_mean,
        "z_std": z_std,
        "feat_mean": feat_mean.tolist(),
        "feat_std": feat_std.tolist(),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "epochs_requested": EPOCHS,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_f1": float(best_val_f1),
        "validation_accuracy": float(val_eval_acc),
        "validation_macro_f1": float(val_eval_f1),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"Saved model -> {model_path}")
    print(f"Saved meta  -> {meta_path}")
